In [ ]:
import pandas as pd

## 1. Create views dataframe and extract datetime components

In [ ]:
# 1) Read feed-views.log into a dataframe called views
views = pd.read_csv(
    "data/feed-views.log",
    sep="\t",                 
    header=None,              # header
    names=["datetime", "user"]  # datetime, user
)

# 2) Convert datetime column to datetime64[ns]
views["datetime"] = pd.to_datetime(views["datetime"])

# 3) Extract year, month, day, hour, minute, second to new columns
views["year"] = views["datetime"].dt.year
views["month"] = views["datetime"].dt.month
views["day"] = views["datetime"].dt.day
views["hour"] = views["datetime"].dt.hour
views["minute"] = views["datetime"].dt.minute
views["second"] = views["datetime"].dt.second

views.head()

## 2. Create daytime column using pd.cut and set user as index

In [ ]:
# Define bins and labels for daytime categories
bins = [-1, 3, 6, 10, 16, 19, 23]
labels = [
    "night",
    "early morning",
    "morning",
    "afternoon",
    "early evening",
    "evening"
]

# Create the 'daytime' column using pd.cut on the 'hour' column
views["daytime"] = pd.cut(
    views["hour"],
    bins=bins,
    labels=labels
)

# Set 'user' as the index
views = views.set_index("user")

views[["datetime", "hour", "daytime"]].head()

## 3. Count elements in dataframe and per daytime category

In [ ]:
# Number of non-null elements in each column
elements_per_column = views.count()

# If you want the total number of elements (cells) in the dataframe:
total_elements = elements_per_column.sum()

elements_per_column, total_elements

In [2]:
# Number of elements in each daytime category
daytime_counts = views["daytime"].value_counts()

daytime_counts

NameError: name 'views' is not defined

## 4. Sort values by hour, minute and second

In [ ]:
# Sort by hour, then minute, then second (all ascending)
views_sorted = views.sort_values(
    by=["hour", "minute", "second"],
    ascending=True
)

views_sorted[["datetime", "hour", "minute", "second"]].head()

## 5. Min/max hour, night/morning filters, and modes

In [ ]:
# Global min and max for hour
min_hour = views["hour"].min()
max_hour = views["hour"].max()

min_hour, max_hour

In [ ]:
# Rows where daytime is 'night'
night_rows = views[views["daytime"] == "night"]
night_max_hour = night_rows["hour"].max()

# Example user who visited at that maximum night hour
night_example_user = night_rows[night_rows["hour"] == night_max_hour].index[0]

# Rows where daytime is 'morning'
morning_rows = views[views["daytime"] == "morning"]
morning_min_hour = morning_rows["hour"].min()

# Example user who visited at that minimum morning hour
morning_example_user = morning_rows[morning_rows["hour"] == morning_min_hour].index[0]

night_max_hour, night_example_user, morning_min_hour, morning_example_user

In [ ]:
# Mode for hour and for daytime
hour_mode = views["hour"].mode()[0]
daytime_mode = views["daytime"].mode()[0]

hour_mode, daytime_mode

## 6. Three earliest and latest times with usernames (nsmallest / nlargest)

In [ ]:
# 3 earliest times (by hour, minute, second)
earliest_rows = views.nsmallest(
    3,
    columns=["hour", "minute", "second"]
)

# 3 latest times (by hour, minute, second)
latest_rows = views.nlargest(
    3,
    columns=["hour", "minute", "second"]
)

earliest_rows[["datetime"]], latest_rows[["datetime"]]

## 7. Describe statistics and interquartile range (IQR) for hour

In [ ]:
# Describe numeric columns
views_describe = views.describe()

views_describe

In [ ]:
# Extract Q1 (25%) and Q3 (75%) for hour and compute IQR
hour_desc = views["hour"].describe()

q1 = hour_desc["25%"]
q3 = hour_desc["75%"]
iqr = q3 - q1

iqr